# <center> UE23CS342AA2 - Data Analytics </center>

# <center> Worksheet 4b: Content-Based Recommender Systems </center>

<center> Designed by Motamarri Sai Sathvik, Sruthi Sivakumar & Rishi Gupta PESU-26 </center>

<br>

- Student name: Krishank Kureti
- SRN: PES2UG23CS285
- Section: 'E'

## Introduction

You are a Data Analyst intern at 'BrandConnect', a premier influencer marketing agency. The agency wants to leverage data science to move beyond manual influencer discovery. Your task is to develop a recommender system that can identify and recommend social media influencers who are best aligned with a brand’s target audience and campaign goals.

# Details of the dataset:

For this task, you'll use a synthetic dataset of influencer backstories. It has been augmented with two additional features: 'Content Type' and 'Include Platforms'.

### Columns in this dataset:

- **'Name'**: The influencer's name.
- **'Age'**: The influencer's age.
- **'Region'**: The geographic region of the influencer (e.g., North America, Europe).
- **'MBTI Personality'**: The influencer's personality type.
- **'Lifestyle'**: Keywords describing the influencer's lifestyle and content niche (e.g., fashion, gaming, tech).
- **'Bio'**: A short biography.
- **'Content Type'**: The primary format of their content (Text, Image, Video, Mixed).
- **'Include Platforms'**: A list of social media platforms the influencer is active on (Instagram, Facebook, Youtube, TikTok).

## Core Concepts:

### Content-Based Filtering

Content-Based Filtering is a type of recommender system that recommends items based on their attributes. The core idea is to recommend items that are similar to other items a user has liked in the past.

In our scenario, the 'user' is a brand looking for influencers. The 'items' are the influencers in our dataset. The 'liking' is determined by how well an influencer's attributes (content, demographics, platform) match the brand's campaign brief (target audience, desired content style).

We will create a profile for each influencer based on their attributes and compare it against a brand's desired profile to find the best matches.

## 1. Setup: Install and Load Packages

First, we need to install the necessary libraries.

In [2]:
# Install the necessary libraries using the following command:
# !pip install pandas scikit-learn

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## 2. Load and Explore the Data

Load the `influencer_data_augmented.jsonl` dataset. Remember that `.jsonl` files are JSON files where each line is a separate JSON object.

In [11]:
import pandas as pd
influencers = pd.read_json("/kaggle/input/content-based-recommender-system-dataset/influencer_data_augmented.jsonl", lines=True)
print("Head of dataset:")
print(influencers.head())
print("\nDataset Info:")
print(influencers.info())
print("\nMissing Values:")
print(influencers.isnull().sum())
print("\nDescribe:")
print(influencers.describe())
print("\nUnique counts:")
print(influencers.apply(lambda col: col.nunique() if col.dtype != 'object' or col.apply(lambda x: isinstance(x, list)).sum() == 0 else "List values"))

Head of dataset:
               Name  Age     Sex Country of Origin State or Province  \
0     Andre Mcclain   23    male               USA    South Carolina   
1    Gregory Joseph   19    male               USA              Iowa   
2      William Wall   23    male               USA            Oregon   
3  Virginia Chapman   24  female               USA      Pennsylvania   
4    Deborah Potter   33  female            Canada            Quebec   

  Education Level MBTI Personality     Lifestyle  \
0     High School             INFP  architecture   
1     High School             ISFP           diy   
2     High School             ISFP          pets   
3        Bachelor             ISFJ       cooking   
4        Bachelor             ESTP       dancing   

                                           Backstory Content Type  \
0  Andre Mcclain grew up in a small town in South...        Video   
1  Gregory Joseph was born and raised in a small ...        Mixed   
2  William Wall was born and r

## 3. EDA and Preprocessing

**(1 point)**

State some observations that you made after performing EDA.

Feel free to remove any columns you think are not necessary for the following problems.

Some hints to help you get started:
- Check for null values and duplicates.
- Examine the data types of each column.
- For this task, we will focus on 'Age', 'Region', 'Lifestyle', 'Content Type', and 'Include Platforms'. Columns like 'Bio', 'Name', and 'MBTI Personality' can be dropped for simplicity.

After looking through the dataset, everything seems pretty clean — no missing values, no obvious duplicates, and the data types are all consistent. The only slightly messy part is the platform lists, since they're stored as actual lists instead of strings, but that’s manageable.


In [12]:
influencers_cleaned = influencers.drop(columns=[
    'Name',
    'Sex',
    'State or Province',
    'Education Level',
    'MBTI Personality',
    'Backstory'
])
print(influencers_cleaned.head())
print(influencers_cleaned.info())

   Age Country of Origin     Lifestyle Content Type  \
0   23               USA  architecture        Video   
1   19               USA           diy        Mixed   
2   23               USA          pets        Mixed   
3   24               USA       cooking        Mixed   
4   33            Canada       dancing        Video   

             Include Platforms  
0          [TikTok, Instagram]  
1  [Youtube, Facebook, TikTok]  
2          [TikTok, Instagram]  
3                     [TikTok]  
4                   [Facebook]  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32890 entries, 0 to 32889
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                32890 non-null  int64 
 1   Country of Origin  32890 non-null  object
 2   Lifestyle          32890 non-null  object
 3   Content Type       32890 non-null  object
 4   Include Platforms  32890 non-null  object
dtypes: int64(1), object(4)
memory

## Problem 1: Feature Engineering

**(2 points)**

To compare influencers, we need to convert their attributes into a single, consistent representation.

Create a new column called `'feature_soup'` in your dataframe. This column should be a string that combines the 'Region', 'Lifestyle', 'Content Type', and 'Include Platforms' for each influencer. This unified string will serve as the basis for our content-based matching.

In [13]:
influencers_cleaned['feature_soup'] = (
    influencers_cleaned['Country of Origin'].astype(str) + " " +
    influencers_cleaned['Lifestyle'].astype(str) + " " +
    influencers_cleaned['Content Type'].astype(str) + " " +
    influencers_cleaned['Include Platforms'].astype(str)
)
print(influencers_cleaned[['Age', 'Country of Origin', 'Lifestyle', 'Content Type', 'Include Platforms', 'feature_soup']].head())

   Age Country of Origin     Lifestyle Content Type  \
0   23               USA  architecture        Video   
1   19               USA           diy        Mixed   
2   23               USA          pets        Mixed   
3   24               USA       cooking        Mixed   
4   33            Canada       dancing        Video   

             Include Platforms  \
0          [TikTok, Instagram]   
1  [Youtube, Facebook, TikTok]   
2          [TikTok, Instagram]   
3                     [TikTok]   
4                   [Facebook]   

                                      feature_soup  
0   USA architecture Video ['TikTok', 'Instagram']  
1  USA diy Mixed ['Youtube', 'Facebook', 'TikTok']  
2           USA pets Mixed ['TikTok', 'Instagram']  
3                     USA cooking Mixed ['TikTok']  
4                Canada dancing Video ['Facebook']  


## Problem 2: Building the Recommender

**(4 points: 2 per task)**

Now, let's build the core of the recommender system.

1.  **Vectorize the Features**: Use `TfidfVectorizer` from `scikit-learn` to transform the `'feature_soup'` column into a matrix of TF-IDF features. TF-IDF helps in giving more weight to niche, important keywords and less weight to common ones.

2.  **Define a Brand Profile & Recommend**: A brand provides you with their campaign requirements:
    - **Target Age Group**: 20-30 years
    - **Target Region**: 'North America'
    - **Preferred Content Type**: 'Video'
    - **Preferred Platforms**: 'Youtube' and 'Tiktok'

    Your task is to recommend the **top 5 influencers** that best match this profile. Remember to filter by age first, then use cosine similarity on the TF-IDF vectors to find the best matches based on the other criteria.

In [14]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(influencers_cleaned['feature_soup'])
brand_profile_text = (
    "North America lifestyle Video " +
    "Youtube TikTok"
)

brand_vec = vectorizer.transform([brand_profile_text])
age_filtered = influencers_cleaned[(influencers_cleaned['Age'] >= 20) &
                                   (influencers_cleaned['Age'] <= 30)]

age_filtered_indices = age_filtered.index
similarities = cosine_similarity(brand_vec, tfidf_matrix[age_filtered_indices]).flatten()
top5_indices = similarities.argsort()[::-1][:5]
top5_influencers = age_filtered.iloc[top5_indices]
print("Top 5 Influencer Matches for the Brand:")
print(top5_influencers[['Age', 'Country of Origin', 'Lifestyle', 'Content Type', 'Include Platforms']])

Top 5 Influencer Matches for the Brand:
       Age Country of Origin  Lifestyle Content Type  Include Platforms
4725    20               USA  lifestyle        Video  [Youtube, TikTok]
30149   20               USA  lifestyle        Video  [TikTok, Youtube]
23752   21               USA  lifestyle        Video  [TikTok, Youtube]
1250    20               USA  lifestyle        Video  [Youtube, TikTok]
24148   23               USA  lifestyle        Video  [TikTok, Youtube]


## Problem 3: Encapsulating the Logic

**(2 points)**

To make your system reusable, create a Python function `get_recommendations(brand_profile, df, n=5)` that takes a brand profile dictionary, the dataframe, and the number of recommendations `n` as input. It should perform all the steps from Problem 2 and return a dataframe of the top `n` recommended influencers.

In [15]:
def get_recommendations(brand_profile, df, n=5):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df['feature_soup'])
    brand_profile_text = (
        brand_profile['Region'] + " " +
        brand_profile['Lifestyle'] + " " +
        brand_profile['Content Type'] + " " +
        " ".join(brand_profile['Platforms'])
    )
    
    brand_vec = vectorizer.transform([brand_profile_text])
    min_age, max_age = brand_profile['Age Range']
    age_filtered = df[(df['Age'] >= min_age) & (df['Age'] <= max_age)]
    
    filtered_indices = age_filtered.index
    similarities = cosine_similarity(brand_vec, tfidf_matrix[filtered_indices]).flatten()
    top_indices = similarities.argsort()[::-1][:n]
    top_influencers = age_filtered.iloc[top_indices]
    
    return top_influencers

brand_profile = {
    "Age Range": (20, 30),
    "Region": "North America",
    "Lifestyle": "lifestyle",
    "Content Type": "Video",
    "Platforms": ["Youtube", "TikTok"]
}

recommendations = get_recommendations(brand_profile, influencers_cleaned, n=5)
print(recommendations)

       Age Country of Origin  Lifestyle Content Type  Include Platforms  \
4725    20               USA  lifestyle        Video  [Youtube, TikTok]   
30149   20               USA  lifestyle        Video  [TikTok, Youtube]   
23752   21               USA  lifestyle        Video  [TikTok, Youtube]   
1250    20               USA  lifestyle        Video  [Youtube, TikTok]   
24148   23               USA  lifestyle        Video  [TikTok, Youtube]   

                                    feature_soup  
4725   USA lifestyle Video ['Youtube', 'TikTok']  
30149  USA lifestyle Video ['TikTok', 'Youtube']  
23752  USA lifestyle Video ['TikTok', 'Youtube']  
1250   USA lifestyle Video ['Youtube', 'TikTok']  
24148  USA lifestyle Video ['TikTok', 'Youtube']  


## Problem 4: Shortcomings & Improvements

**(1 point)**

What are the main shortcomings of this content-based filtering approach? Suggest one modification or additional feature that could be introduced to achieve better, more nuanced results.

A big downside of this content-based approach is that it only compares influencers to the brand’s stated preferences it doesn’t really understand deeper patterns or how different influencers actually perform. One simple improvement would be to incorporate performance metrics such as follower count, engagement rate, or growth trends into the feature vector. Even basic weighting toward high-quality influencers would make the recommendations more realistic and useful.

---
With that, our journey through the world of influencer analytics comes to an end, and we return to the familiar realm of data.

We hope this exercise gave you a practical glimpse into how data science powers decision-making in modern marketing, especially in the fast-evolving space of influencer discovery.

You've successfully built a foundational recommender system tailored for influencer marketing—combining structured profiles, personality insights, and platform data. Great job!

Keep exploring, keep experimenting—and most importantly, keep learning :)